<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Estaciones Meteorológicas Ficticias y sus Datos Torcidos 🌦️
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 01
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/01_Descriptiva_Visualizacion_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 01: Estadística Descriptiva y Visualización** ([README del módulo](../01%20-%20Estadistica%20Descriptiva%20y%20Visualizacion/README.md)) con un problema **nuevo**: el registro diario de tres **estaciones meteorológicas ficticias** (temperatura, lluvia, humedad y un índice de estrés térmico), con lecturas defectuosas y variables muy asimétricas.

Al terminar serás capaz de:

1. **Resumir con robustez** una variable asimétrica: media, mediana, media recortada y winsorizada, MAD e IQR, **cuantiles de dos definiciones** (calculados a mano) y tres reglas para señalar **valores atípicos**.
2. **Medir la forma** (asimetría y curtosis) y **simetrizar con Box-Cox**: la log-verosimilitud perfil, el $\hat\lambda$ y su intervalo de verosimilitud, y comprobar la mejora con un gráfico Q-Q numérico.
3. **Cuantificar la asociación**: Pearson frente a Spearman, una relación en forma de U que ningún coeficiente lineal ve, la **influencia de un solo dato** y la **V de Cramér** para variables categóricas.
4. **Graficar con honestidad**: número de clases por tres reglas, la función de distribución empírica (ECDF), el **factor de mentira** de un eje truncado y una figura de dos paneles bien rotulada.

> 📚 **Repasa si lo necesitas:** [Tendencia central y posición](../01%20-%20Estadistica%20Descriptiva%20y%20Visualizacion/00_Medidas_de_Tendencia_Central_y_Posicion.ipynb) · [Dispersión, forma y asociación](../01%20-%20Estadistica%20Descriptiva%20y%20Visualizacion/01_Medidas_de_Dispersion_Forma_y_Asociacion.ipynb) · [Visualización de datos estadísticos](../01%20-%20Estadistica%20Descriptiva%20y%20Visualizacion/02_Visualizacion_de_Datos_Estadisticos.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija; las estaciones y sus cifras son **inventadas**). Bibliotecas: `numpy`, `pandas`, `scipy` y `matplotlib`. Todo el cuaderno corre en pocos segundos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

SEMILLA = 2025


def generar_estaciones(semilla=SEMILLA, n=600):
    """Registro diario ficticio de tres estaciones (A: 2 600 m, B: 1 800 m, C: 3 100 m).

    precip_mm: 0 en días secos; cuatro lecturas de lluvia fueron digitadas 10 veces más grandes (error sembrado).
    estres: índice de estrés térmico, mínimo cerca de la temperatura media y creciente hacia ambos lados (forma de U).
    """
    rng = np.random.default_rng(semilla)
    estacion = rng.choice(["A", "B", "C"], n, p=[0.40, 0.35, 0.25])
    alt = pd.Series(estacion).map({"A": 2600, "B": 1800, "C": 3100}).to_numpy() + rng.normal(0, 20, n)
    temp = 32 - 0.0065 * alt + rng.normal(0, 1.8, n)
    llueve = rng.random(n) < pd.Series(estacion).map({"A": 0.55, "B": 0.75, "C": 0.35}).to_numpy()
    precip = np.where(llueve, rng.lognormal(1.3, 0.8, n), 0.0)
    mal = rng.choice(np.flatnonzero(llueve), 4, replace=False)
    precip[mal] *= 10
    humedad = np.clip(55 + 12 * llueve + 4 * np.log1p(precip) + rng.normal(0, 6, n), 20, 100)
    estres = (temp - temp.mean()) ** 2 / 4 + rng.normal(0, 0.6, n)
    return pd.DataFrame({"estacion": estacion, "altitud_m": alt.round(0), "temp_c": temp.round(1), "precip_mm": precip.round(1),
                         "humedad": humedad.round(1), "estres": estres.round(2)})


estaciones = generar_estaciones()
print(f"Registro ficticio: {len(estaciones)} días-estación | días con lluvia: {int((estaciones['precip_mm'] > 0).sum())}")
print(estaciones.head(3).to_string(index=False))

---
## Reto 1: Resumir la lluvia sin dejarse engañar 🌧️

**Contexto:** la lluvia de los días lluviosos (`lluvia`) es muy asimétrica y tiene lecturas defectuosas. La media se deja arrastrar; las medidas robustas, no. Vas a comparar resúmenes, a calcular cuantiles **a mano** con dos definiciones y a contar cuántos valores señala cada regla de atípicos.

**Enunciado** (con $n$ = número de días lluviosos y $x_{(1)}\le\dots\le x_{(n)}$ los datos ordenados)
1. `media`, `mediana`, `media_rec` (recortada: descarta los $k=\lfloor 0.10\,n\rfloor$ menores y los $k$ mayores) y `media_wins` (**winsorizada**: reemplaza los $k$ menores por $x_{(k+1)}$ y los $k$ mayores por $x_{(n-k)}$).
2. `desv` (desviación estándar, `ddof=1`), `mad_norm` $=1.4826\cdot\text{mediana}(|x_i-\text{mediana}|)$ e `iqr` $=Q_3-Q_1$ con la definición lineal de NumPy.
3. `cuantil_lineal(x, p)`: posición $h=(n-1)p$ (desde 0) e interpolación lineal entre vecinos. `cuantil_weibull(x, p)`: posición $(n+1)p$ contando desde 1, acotada a $[1,n]$, con interpolación lineal.
4. `vallas`: `(Q1 - 1.5·IQR, Q3 + 1.5·IQR)`. `n_tukey`: cuántos valores caen **fuera** de las vallas; `n_zrob`: cuántos tienen $|(x_i-\text{mediana})/\text{mad\_norm}|>3.5$; `n_zclasico`: cuántos tienen $|(x_i-\bar x)/s|>3$.

**Criterios de aceptación:** coinciden con NumPy/SciPy; la media supera a la mediana; la recortada y la winsorizada quedan más cerca de la mediana que la media; y el criterio clásico señala **menos** valores que el robusto y que Tukey (los atípicos inflan la desviación y se «enmascaran»).

**Puntaje sugerido:** 20 puntos.

In [ ]:
lluvia = estaciones.loc[estaciones["precip_mm"] > 0, "precip_mm"].to_numpy()
n_ll = len(lluvia)

# TODO: completa
media = mediana = media_rec = media_wins = None   # rec y wins: 10 % por cada lado
desv = mad_norm = iqr = None


def cuantil_lineal(x, p):
    # TODO: tipo 7 (el de NumPy por defecto): h = (n - 1) * p desde 0, interpolación lineal entre vecinos
    return None


def cuantil_weibull(x, p):
    # TODO: tipo 6: posición (n + 1) * p desde 1, acotada a [1, n], interpolación lineal
    return None


vallas = None        # (inferior, superior) de Tukey con tu cuantil lineal
n_tukey = n_zrob = n_zclasico = None

In [ ]:
# Autoverificación del Reto 1 (no modificar)
_x = lluvia
assert media is not None and abs(media - _x.mean()) < 1e-9 and abs(mediana - np.median(_x)) < 1e-9, "media y mediana de `lluvia` (np.mean, np.median)."
assert abs(media_rec - stats.trim_mean(_x, 0.10)) < 1e-9, "media_rec: ordena, quita los k = int(0.10 * n) menores y los k mayores y promedia el resto."
assert abs(media_wins - stats.mstats.winsorize(_x, limits=(0.10, 0.10)).mean()) < 1e-9, "media_wins: cambia los k menores por el (k+1)-ésimo menor y los k mayores por el (k+1)-ésimo mayor (np.clip con esos dos valores) y promedia."
assert abs(desv - _x.std(ddof=1)) < 1e-9 and abs(mad_norm - stats.median_abs_deviation(_x, scale="normal")) < 1e-3, "desv usa ddof=1 y mad_norm = 1.4826 * mediana(|x - mediana|)."
assert abs(iqr - (np.percentile(_x, 75) - np.percentile(_x, 25))) < 1e-9, "iqr = Q3 - Q1 con cuantiles lineales (los de np.percentile)."
for _p in (0, 0.1, 0.25, 0.5, 0.9, 1):
    assert abs(cuantil_lineal(_x, _p) - np.quantile(_x, _p)) < 1e-9, f"cuantil_lineal(x, {_p}) debe coincidir con np.quantile(x, {_p}): h = (n - 1) * p, parte entera e interpolación con la fracción."
    assert abs(cuantil_weibull(_x, _p) - np.quantile(_x, _p, method="weibull")) < 1e-9, f"cuantil_weibull(x, {_p}): posición (n + 1) * p desde 1, acotada a [1, n]; interpola entre el vecino inferior y el superior."
_q1, _q3 = np.percentile(_x, [25, 75])
assert abs(vallas[0] - (_q1 - 1.5 * (_q3 - _q1))) < 1e-9 and abs(vallas[1] - (_q3 + 1.5 * (_q3 - _q1))) < 1e-9, "vallas = (Q1 - 1.5·IQR, Q3 + 1.5·IQR)."
_tk = int(((_x < vallas[0]) | (_x > vallas[1])).sum())
_zr = int((np.abs((_x - np.median(_x)) / stats.median_abs_deviation(_x, scale="normal")) > 3.5).sum())
_zc = int((np.abs((_x - _x.mean()) / _x.std(ddof=1)) > 3).sum())
assert (n_tukey, n_zrob, n_zclasico) == (_tk, _zr, _zc), f"Esperaba n_tukey={_tk}, n_zrob={_zr}, n_zclasico={_zc}: revisa los umbrales (3.5 con mediana y MAD escalada; 3 con media y desviación)."
assert media > mediana and abs(media_rec - mediana) < abs(media - mediana) and abs(media_wins - mediana) < abs(media - mediana), "La media debe superar a la mediana, y las versiones recortada y winsorizada deben quedar más cerca de la mediana."
assert n_zclasico < n_zrob and n_zclasico < n_tukey, "El criterio clásico debe señalar menos valores que el robusto y que Tukey: los valores extremos inflan la desviación y enmascaran a los demás."
print(f"✅ Reto 1 superado: media {media:.2f}, recortada {media_rec:.2f}, winsorizada {media_wins:.2f}, mediana {mediana:.2f} mm | atípicos: Tukey {n_tukey}, z robusto {n_zrob}, z clásico {n_zclasico}")

---
## Reto 2: Simetrizar la lluvia con Box-Cox 🔔

**Contexto:** con la misma `lluvia`, ahora te interesa la **forma**: cuán asimétrica es y qué potencia $\lambda$ de la familia de Box-Cox
$$y_\lambda=\begin{cases}\dfrac{x^\lambda-1}{\lambda}, & \lambda\neq 0,\\[2mm] \ln x, & \lambda=0,\end{cases}$$
la vuelve más parecida a una normal. La **log-verosimilitud perfil** (con $n$ datos y $\hat\sigma^2_\lambda$ la varianza, `ddof=0`, de $y_\lambda$) es
$$\ell(\lambda)=(\lambda-1)\sum_i\ln x_i-\frac{n}{2}\ln\hat\sigma^2_\lambda .$$

**Enunciado**
1. `g1` (asimetría de momentos $m_3/m_2^{3/2}$ con $m_k=\frac1n\sum(x_i-\bar x)^k$), `G1` (la versión ajustada $g_1\sqrt{n(n-1)}/(n-2)$) y `g2` (curtosis en exceso $m_4/m_2^2-3$).
2. `llf_boxcox(lmbda, x)` y, sobre la rejilla `lambdas`, el vector `llfs`; `lam_hat` (el $\lambda$ de la rejilla que maximiza).
3. `ic_lambda`: `(lo, hi)`, el menor y el mayor $\lambda$ de la rejilla con $\ell(\lambda)\ge\ell(\hat\lambda)-1.92$ (intervalo de verosimilitud aproximado al 95 %, porque $\chi^2_{1,0.95}/2\approx1.92$).
4. `x_bc`: los datos transformados con `lam_hat`; `g1_antes` y `g1_despues` (asimetría `g1`), y `r_antes`, `r_despues`: la **correlación Q-Q** entre los datos ordenados y los cuantiles normales $\Phi^{-1}\!\big((i-0.5)/n\big)$.

**Criterios de aceptación:** las medidas de forma coinciden con SciPy; tu $\hat\lambda$ difiere de `stats.boxcox` en menos de 0.011; el intervalo contiene a $\hat\lambda$; la asimetría cae a menos de un tercio de la original y la correlación Q-Q sube a más de 0.98.

**Puntaje sugerido:** 25 puntos.

In [ ]:
lambdas = np.arange(-200, 201) / 100      # rejilla de -2 a 2 con paso 0.01 (incluye el 0 exacto)

# TODO: completa
g1 = G1 = g2 = None


def llf_boxcox(lmbda, x):
    # TODO: log-verosimilitud perfil; para lmbda == 0 usa el logaritmo
    return None


llfs = None          # arreglo con llf_boxcox(l, lluvia) para cada l de `lambdas`
lam_hat = None
ic_lambda = None     # (lo, hi)
x_bc = None          # lluvia transformada con lam_hat
g1_antes = g1_despues = None
r_antes = r_despues = None

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_x = lluvia
assert g1 is not None and abs(g1 - stats.skew(_x)) < 1e-9 and abs(G1 - stats.skew(_x, bias=False)) < 1e-9, "g1 = m3 / m2**1.5 (momentos con divisor n); G1 = g1 * sqrt(n(n-1)) / (n-2)."
assert abs(g2 - stats.kurtosis(_x)) < 1e-9, "g2 = m4 / m2**2 - 3 (curtosis en exceso, momentos con divisor n)."
assert len(llfs) == len(lambdas), "llfs debe tener un valor por cada lambda de la rejilla."
for _l in (-1.0, -0.37, 0.0, 0.5, 1.0):
    assert abs(llf_boxcox(_l, _x) - stats.boxcox_llf(_l, _x)) < 1e-6, f"llf_boxcox({_l}, x) debe coincidir con stats.boxcox_llf: (l - 1)·Σ ln x - (n/2)·ln(var de y_l) con var de ddof=0 y, si l = 0, y = ln x."
assert np.allclose(llfs, [stats.boxcox_llf(l, _x) for l in lambdas], atol=1e-6), "llfs: evalúa llf_boxcox en cada elemento de `lambdas`."
_sc = stats.boxcox(_x)[1]
assert abs(lam_hat - _sc) < 0.011, f"lam_hat debe ser el lambda de la rejilla donde llfs es máxima (cerca de {_sc:.3f})."
_dn = lambdas[llfs >= llfs.max() - 1.92]
assert ic_lambda is not None and abs(ic_lambda[0] - _dn.min()) < 1e-9 and abs(ic_lambda[1] - _dn.max()) < 1e-9 and ic_lambda[0] <= lam_hat <= ic_lambda[1], "ic_lambda = (menor, mayor) lambda de la rejilla con llf >= max(llf) - 1.92."
_y = np.log(_x) if lam_hat == 0 else (_x ** lam_hat - 1) / lam_hat
assert x_bc is not None and np.allclose(x_bc, _y), "x_bc = (x**lam_hat - 1)/lam_hat (o log(x) si lam_hat == 0)."
assert abs(g1_antes - stats.skew(_x)) < 1e-9 and abs(g1_despues - stats.skew(_y)) < 1e-9, "g1_antes y g1_despues: la asimetría g1 de los datos originales y de x_bc."
assert abs(g1_despues) < abs(g1_antes) / 3, "La transformación debería reducir la asimetría a menos de un tercio."
_n = len(_x); _t = stats.norm.ppf((np.arange(1, _n + 1) - 0.5) / _n)
assert abs(r_antes - np.corrcoef(np.sort(_x), _t)[0, 1]) < 1e-9 and abs(r_despues - np.corrcoef(np.sort(_y), _t)[0, 1]) < 1e-9, "r = np.corrcoef de los datos ORDENADOS con norm.ppf((i - 0.5)/n), i = 1..n."
assert r_despues > 0.98 > r_antes, "Después de Box-Cox el gráfico Q-Q debe quedar casi recto (r > 0.98), cosa que no ocurría antes."
print(f"✅ Reto 2 superado: g1 {g1:.2f} → {g1_despues:.2f} | lambda = {lam_hat:.2f} (IC {ic_lambda[0]:.2f} a {ic_lambda[1]:.2f}) | r Q-Q {r_antes:.3f} → {r_despues:.3f}")

---
## Reto 3: ¿Qué ve cada coeficiente de asociación? 🔗

**Contexto:** en `estaciones` la humedad sube con la lluvia, pero de forma muy desigual; el índice de estrés térmico `estres` depende de la temperatura **en forma de U**; y un sensor defectuoso (`punto`) añade un dato absurdo a una muestra pequeña (`peque`). Cada coeficiente mira algo distinto.

**Enunciado**
1. `c_pearson` y `c_spearman`: matrices de correlación 4×4 de las columnas `cols` (`DataFrame`). `par_disc`: el par `(a, b)` con `a < b` alfabéticamente (distintos) que tiene mayor $|\rho_S-r|$.
2. `r_p_u`: Pearson entre `temp_c` y `estres`; `r2_cuad`: $R^2$ del ajuste cuadrático de `estres` sobre `temp_c` (`np.polyfit`, grado 2).
3. `r_p0`, `r_s0`: Pearson y Spearman entre `temp_c` y `humedad` en `peque`; `r_p1`, `r_s1`: lo mismo tras **añadir** `punto` (una fila). `delta_p` y `delta_s`: el cambio absoluto de cada uno.
4. `tabla_ct`: tabla de contingencia (`pd.crosstab`) de `estacion` frente a `categoria_lluvia`; `v_cramer(tabla)` desde cero: $V=\sqrt{\chi^2/\big(n\min(r-1,c-1)\big)}$ con $\chi^2$ sin corrección; `v` su valor en `tabla_ct`.

**Criterios de aceptación:** coinciden con pandas/SciPy; Pearson ≈ 0 pero $R^2>0.8$ en la relación en U; `delta_p` más de tres veces `delta_s`; y `v` coincide con `stats.contingency.association`.

**Puntaje sugerido:** 20 puntos.

In [ ]:
cols = ["humedad", "precip_mm", "temp_c", "estres"]
peque = estaciones.iloc[:40][["temp_c", "humedad"]]
punto = pd.DataFrame({"temp_c": [45.0], "humedad": [100.0]})            # lectura de un sensor defectuoso
categoria_lluvia = pd.cut(estaciones["precip_mm"], [-0.1, 0, 10, np.inf], labels=["seco", "ligera", "fuerte"])

# TODO: completa
c_pearson = c_spearman = None
par_disc = None
r_p_u = r2_cuad = None
r_p0 = r_s0 = r_p1 = r_s1 = None
delta_p = delta_s = None


def v_cramer(tabla):
    # TODO: V de Cramér desde cero (chi-cuadrado SIN corrección de Yates)
    return None


tabla_ct = None
v = None

In [ ]:
# Autoverificación del Reto 3 (no modificar)
assert c_pearson is not None and np.allclose(c_pearson.to_numpy(), estaciones[cols].corr().to_numpy()), "c_pearson = estaciones[cols].corr()."
assert np.allclose(c_spearman.to_numpy(), estaciones[cols].corr(method="spearman").to_numpy()), "c_spearman = estaciones[cols].corr(method='spearman')."
_d = (estaciones[cols].corr(method="spearman") - estaciones[cols].corr()).abs()
_pares = [(a, b) for a in cols for b in cols if a < b]
assert par_disc == max(_pares, key=lambda ab: _d.loc[ab[0], ab[1]]), "par_disc: entre los pares con a < b, el que tenga mayor |Spearman - Pearson| (una cola larga o ceros golpean a Pearson, no a Spearman)."
assert r_p_u is not None and abs(r_p_u - np.corrcoef(estaciones["temp_c"], estaciones["estres"])[0, 1]) < 1e-9 and abs(r_p_u) < 0.35, "r_p_u es el Pearson entre temp_c y estres: pequeño aunque la dependencia sea fuerte."
_c = np.polyfit(estaciones["temp_c"], estaciones["estres"], 2); _r = estaciones["estres"] - np.polyval(_c, estaciones["temp_c"])
_r2 = 1 - (_r ** 2).sum() / ((estaciones["estres"] - estaciones["estres"].mean()) ** 2).sum()
assert abs(r2_cuad - _r2) < 1e-9 and r2_cuad > 0.8, "r2_cuad = 1 - SCE/SCT del ajuste np.polyfit(temp_c, estres, 2): una U se explica muy bien con una parábola."
_m = pd.concat([peque, punto], ignore_index=True)
assert abs(r_p0 - peque.corr().iloc[0, 1]) < 1e-9 and abs(r_s0 - peque.corr(method="spearman").iloc[0, 1]) < 1e-9, "r_p0 y r_s0: Pearson y Spearman de `peque` antes de añadir el punto."
assert abs(r_p1 - _m.corr().iloc[0, 1]) < 1e-9 and abs(r_s1 - _m.corr(method="spearman").iloc[0, 1]) < 1e-9, "r_p1 y r_s1: lo mismo con el punto añadido (pd.concat([peque, punto]))."
assert abs(delta_p - abs(r_p1 - r_p0)) < 1e-12 and abs(delta_s - abs(r_s1 - r_s0)) < 1e-12 and delta_p > 3 * delta_s, "delta_p y delta_s son |r1 - r0|; un solo dato extremo debe mover a Pearson más de tres veces lo que mueve a Spearman."
_t = pd.crosstab(estaciones["estacion"], categoria_lluvia)
assert tabla_ct is not None and np.array_equal(tabla_ct.to_numpy(), _t.to_numpy()), "tabla_ct = pd.crosstab(estaciones['estacion'], categoria_lluvia)."
assert abs(v_cramer(_t) - stats.contingency.association(_t, method="cramer", correction=False)) < 1e-9, "v_cramer: esperados = (total fila × total columna)/total; chi2 = Σ (O-E)²/E; V = sqrt(chi2 / (n · min(filas-1, columnas-1)))."
assert abs(v - stats.contingency.association(_t, method="cramer", correction=False)) < 1e-9 and 0 < v < 1, "v = v_cramer(tabla_ct)."
print(f"✅ Reto 3 superado: par más discrepante {par_disc} | U: Pearson {r_p_u:.2f} pero R² cuadrático {r2_cuad:.2f} | un dato mueve Pearson {delta_p:.2f} y Spearman {delta_s:.2f} | V de Cramér = {v:.2f}")

---
## Reto 4: Un gráfico honesto de la temperatura 📊

**Contexto:** para el informe de la red de estaciones debes mostrar la distribución de `temp_c`. Antes de graficar decides cuántas clases usar, calculas la **ECDF** (que no depende de ninguna clase) y revisas cuánto «miente» un gráfico de barras con el eje truncado.

**Enunciado**
1. `k_sturges` $=\lceil\log_2 n+1\rceil$, `k_scott` $=\lceil R/h\rceil$ con $h=3.49\,s\,n^{-1/3}$ ($s$ con `ddof=0`) y `k_fd` $=\lceil R/h\rceil$ con $h=2\,\text{IQR}\,n^{-1/3}$ (regla de Freedman-Diaconis); $R$ es el rango de los datos y $n$ = `n_t`.
2. `ecdf(x)` → `(xs, ys)`: valores ordenados e $i/n$ para $i=1,\dots,n$. `p_20`: $\hat F(20)$, la fracción de días con `temp_c` $\le 20$, leída de `(xs, ys)` con `np.searchsorted` (sin volver a comparar `temp`).
3. `factor_mentira(a, b, y_min)`: cociente entre el efecto **mostrado** en un gráfico de barras con eje desde `y_min`, $\dfrac{b-a}{a-y_{\min}}$, y el efecto **real** en los datos, $\dfrac{b-a}{a}$. Calcula `fm` con `a = 54`, `b = 58` (humedades medias) y `y_min = 50`.
4. `fig`: una figura de **dos paneles** (`plt.subplots(1, 2)`): histograma de `temp` con `k_fd` clases y ECDF (`step`). Cada panel con título, rótulos de ejes y unidades.

**Criterios de aceptación:** las reglas coinciden con `np.histogram_bin_edges`; la ECDF termina en 1 y es no decreciente; el factor de mentira es 1 con el eje desde 0 y mucho mayor con el eje truncado; y la figura tiene dos paneles rotulados, con `k_fd` barras en el histograma.

**Puntaje sugerido:** 20 puntos.

In [ ]:
temp = estaciones["temp_c"].to_numpy()
n_t = len(temp)

# TODO: completa
k_sturges = k_scott = k_fd = None


def ecdf(x):
    # TODO: devuelve (xs, ys) con xs ordenado e ys = i / n
    return None, None


xs, ys = ecdf(temp)
p_20 = None


def factor_mentira(a, b, y_min):
    # TODO: efecto mostrado (alturas desde y_min) / efecto real
    return None


fm = None         # factor_mentira(54, 58, 50)

fig = None        # figura de dos paneles: histograma (k_fd clases) y ECDF

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_k = lambda regla: len(np.histogram_bin_edges(temp, bins=regla)) - 1
assert k_sturges == _k("sturges") and k_scott == _k("scott") and k_fd == _k("fd"), f"Esperaba (sturges, scott, fd) = ({_k('sturges')}, {_k('scott')}, {_k('fd')}): ancho h de cada regla y clases = techo(rango / h) (Sturges: techo(log2 n + 1))."
assert xs is not None and len(xs) == n_t and np.all(np.diff(xs) >= 0) and abs(ys[-1] - 1) < 1e-12 and np.all(np.diff(ys) > 0) and abs(ys[0] - 1 / n_t) < 1e-12, "ecdf: xs ordenado (np.sort) e ys = (1..n)/n."
assert p_20 is not None and abs(p_20 - np.mean(temp <= 20)) < 1e-12, "p_20 = F(20): busca con np.searchsorted(xs, 20, side='right') cuántos valores son <= 20 y toma ys en la posición anterior."
assert abs(factor_mentira(54, 58, 0) - 1) < 1e-12 and abs(factor_mentira(10, 12, 5) - ((2 / 5) / (2 / 10))) < 1e-12, "factor_mentira = ((b - a)/(a - y_min)) / ((b - a)/a); con y_min = 0 vale 1."
assert fm is not None and abs(fm - 13.5) < 1e-9, "fm = factor_mentira(54, 58, 50): el eje truncado exagera la diferencia 13.5 veces."
assert fig is not None and len(fig.axes) == 2, "fig debe tener dos paneles: fig, axes = plt.subplots(1, 2)."
for _ax in fig.axes:
    assert _ax.get_title() and _ax.get_xlabel() and _ax.get_ylabel(), "Cada panel necesita título, rótulo del eje x y rótulo del eje y (con unidades)."
assert len(fig.axes[0].patches) == k_fd, f"El primer panel debe ser un histograma con k_fd = {k_fd} clases (ax.hist(temp, bins=k_fd))."
_ys = np.concatenate([np.asarray(l.get_ydata(), float) for l in fig.axes[1].lines] or [np.array([np.nan])])
assert len(fig.axes[1].lines) >= 1 and abs(np.nanmax(_ys) - 1) < 1e-9, "El segundo panel debe dibujar la ECDF (ax.step(xs, ys, where='post')): la curva llega a 1."
print(f"✅ Reto 4 superado: clases Sturges {k_sturges}, Scott {k_scott}, Freedman-Diaconis {k_fd} | F(20 °C) = {p_20:.3f} | factor de mentira = {fm:.1f}")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- Con tus cifras de `media`, `mediana`, `media_rec` y `media_wins`, ¿qué valor reportarías como «lluvia típica de un día lluvioso» y por qué? ¿Por qué `n_zclasico` señala menos valores que `n_zrob` y qué significa que Tukey marque tantos? ¿Todo valor señalado es un error?
- ¿Qué te dice `lam_hat` y su intervalo `ic_lambda` sobre la forma de la lluvia? ¿Qué pierdes al interpretar las medias de datos transformados?
- ¿Por qué Spearman supera a Pearson en tu `par_disc`, y por qué ningún coeficiente lineal detecta la relación de `estres` con `temp_c` aunque `r2_cuad` sea alto? ¿Qué mostraría un gráfico que los coeficientes ocultan?
- ¿Qué te enseña `delta_p` frente a `delta_s` sobre un solo dato defectuoso? ¿Qué hacer con él antes de reportar la correlación?
- ¿Cuántas clases usarías y por qué? ¿Qué decisión de diseño sería deshonesta con `fm` y cuál sería la alternativa?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: medidas robustas, cuantiles a mano con dos definiciones y reglas de atípicos | 20 |
| Reto 2: asimetría, curtosis, Box-Cox por verosimilitud perfil y verificación Q-Q | 25 |
| Reto 3: Pearson, Spearman, relación en U, influencia de un dato y V de Cramér | 20 |
| Reto 4: reglas de clases, ECDF, factor de mentira y figura honesta de dos paneles | 20 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Mi figura tiene títulos, rótulos con unidades y el número de clases justificado.
- [ ] Distingo en mis palabras un valor atípico de un error de medición.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Estadística, Análisis y Representación de Datos</i>
  </p>
</div>